# Sensitivity — Task 2/5: stability weight (real SAC re-runs)

Standalone notebook: run top to bottom on a GPU. Contains the full shared setup (game, networks, inner PPO, SAC, validation, sensitivity module, config) followed by this one sweep only.


# Sensitivity — Task 2 of 5: **stability weight**

Genuine SAC re-runs sweeping `w_stability` with CRN across values. Writes `sensitivity_w_stability_{LABEL}.csv` and its plot.

> All five notebooks write to the same `sensitivity_B5_lean_compliant/` folder, so run them in parallel and combine the CSVs afterward.


## Shared setup (identical across all 5 task notebooks)


In [1]:
import jax, jax.numpy as jnp
from jax import lax
from functools import partial
import numpy as np
import pandas as pd
import itertools, time, os
from scipy.integrate import solve_ivp

MASTER_SEED = 42
np.random.seed(MASTER_SEED)
print("JAX", jax.__version__, "| devices:", jax.devices())


JAX 0.10.1 | devices: [CpuDevice(id=0)]


In [2]:
PAPER_SCALE = True   # <--- set True on GPU for the reported numbers

# Paper-scale settings match the actual calls in the original E3/E4/E7/E8 notebooks:
#   n_runs=100, n_epochs(outer)=200, ppo_episodes(inner)=3000,
#   patience=1000 (early stopping effectively disabled), n_val_seeds=5, warmup_epochs=5.
if PAPER_SCALE:
    CFG = dict(N_RUNS=100, N_OUTER=500, N_EPISODES=3000,
               N_VAL_STARTS=20, PATIENCE=1000, WARMUP=5)
else:
    CFG = dict(N_RUNS=8,   N_OUTER=12,  N_EPISODES=800,
               N_VAL_STARTS=12, PATIENCE=1000, WARMUP=5)

OUTDIR = "results_5player"
os.makedirs(OUTDIR, exist_ok=True)
def save_csv(df, name):
    path = os.path.join(OUTDIR, name)
    df.to_csv(path, index=False)
    print(f"  saved -> {path}  ({len(df)} rows)")
    return path
print("PAPER_SCALE =", PAPER_SCALE, "| config:", CFG)


PAPER_SCALE = True | config: {'N_RUNS': 100, 'N_OUTER': 500, 'N_EPISODES': 3000, 'N_VAL_STARTS': 20, 'PATIENCE': 1000, 'WARMUP': 5}


In [3]:
# ---- strategy labels (VARIABLE count per player: 3 or 4) ----
STRAT = {
 'supplier':    ['high','standard','sub'],                     # 3
 'manufacturer':['fullQC','partQC','minQC','noQC'],            # 4
 'inspector':   ['strict','sampling','slack'],                 # 3
 'distributor': ['accept','reinspect','return','recall'],      # 4
 'regulator':   ['auditH','auditM','auditL','noAudit'],        # 4
}
PLAYERS   = list(STRAT.keys())
N_PLAYERS = len(PLAYERS)
STRAT_COUNTS = [len(STRAT[p]) for p in PLAYERS]
K_MAX = max(STRAT_COUNTS)
print("players:", PLAYERS)
print("strategy counts:", STRAT_COUNTS, "| K_max:", K_MAX)
print("joint pure profiles:", int(np.prod(STRAT_COUNTS)))
print("reduced Jacobian dimension:", sum(k-1 for k in STRAT_COUNTS))

# valid-strategy mask: 1 where a strategy exists, 0 where padded to K_MAX
VALID_MASK = np.zeros((N_PLAYERS, K_MAX))
for i,k in enumerate(STRAT_COUNTS):
    VALID_MASK[i,:k] = 1.0
VALID_MASK = jnp.array(VALID_MASK)


players: ['supplier', 'manufacturer', 'inspector', 'distributor', 'regulator']
strategy counts: [3, 4, 3, 4, 4] | K_max: 4
joint pure profiles: 576
reduced Jacobian dimension: 13


In [4]:
PARAM_NAMES = ['P','Cs_h','Cs_s','Cs_b','Cm_f','Cm_p','Cm_m','Cm_n',
               'Ci_s','Ci_p','Ci_0','Ca_h','Ca_m','Ca_l','Cd_r','Cd_ret','Cd_rec',
               'W','B','F_m','F_s','I','R_p','R_c','Vq','G','Sc']
# Wider, economically-ordered bounds. Vq (quality premium), G (governance revenue), and
# Sc (inspection-service revenue) are game parameters that reward realized quality and oversight.
PARAM_BOUNDS = {
 'P':(120,450),'Cs_h':(60,140),'Cs_s':(35,95),'Cs_b':(12,55),
 'Cm_f':(35,110),'Cm_p':(20,75),'Cm_m':(10,50),'Cm_n':(3,28),
 'Ci_s':(20,90),'Ci_p':(8,45),'Ci_0':(1,15),
 'Ca_h':(20,90),'Ca_m':(12,55),'Ca_l':(5,30),
 'Cd_r':(5,40),'Cd_ret':(12,60),'Cd_rec':(25,95),
 'W':(30,220),'B':(20,180),'F_m':(40,320),'F_s':(20,200),
 'I':(5,70),'R_p':(10,70),'R_c':(50,220),
 'Vq':(20,220),'G':(10,180),'Sc':(10,150),
}
N_PARAMS = len(PARAM_NAMES)
LOWER = jnp.array([PARAM_BOUNDS[n][0] for n in PARAM_NAMES])
UPPER = jnp.array([PARAM_BOUNDS[n][1] for n in PARAM_NAMES])
# Logical ordering constraints enforced by monotone repair after scaling:
#   P > Cs_h ; Cs_h>Cs_s>Cs_b ; Cm_f>Cm_p>Cm_m>Cm_n ; Ci_s>Ci_p>Ci_0 ;
#   Ca_h>Ca_m>Ca_l ; Cd_rec>Cd_ret>Cd_r
_ORDER_GROUPS = [['Cs_h','Cs_s','Cs_b'],['Cm_f','Cm_p','Cm_m','Cm_n'],
                 ['Ci_s','Ci_p','Ci_0'],['Ca_h','Ca_m','Ca_l'],
                 ['Cd_rec','Cd_ret','Cd_r']]
print("theta dimension:", N_PARAMS)

theta dimension: 27


In [5]:
Q_SUP = {'high':0.97,'standard':0.80,'sub':0.45}
Q_MFR = {'fullQC':0.97,'partQC':0.85,'minQC':0.70,'noQC':0.55}
D_INS = {'strict':0.85,'sampling':0.50,'slack':0.10}
D_DIS = {'accept':0.0,'reinspect':0.55,'return':0.85,'recall':1.0}
D_REG = {'auditH':0.55,'auditM':0.35,'auditL':0.18,'noAudit':0.0}
_A_LVL   = {'auditH':1.0,'auditM':0.6,'auditL':0.3,'noAudit':0.0}
_INS_LVL = {'strict':1.0,'sampling':0.55,'slack':0.15}

def build_payoff_tensor(th):
    c_sup={'high':th['Cs_h'],'standard':th['Cs_s'],'sub':th['Cs_b']}
    c_mfr={'fullQC':th['Cm_f'],'partQC':th['Cm_p'],'minQC':th['Cm_m'],'noQC':th['Cm_n']}
    c_ins={'strict':th['Ci_s'],'sampling':th['Ci_p'],'slack':th['Ci_0']}
    c_reg={'auditH':th['Ca_h'],'auditM':th['Ca_m'],'auditL':th['Ca_l'],'noAudit':0.0}
    c_dis={'accept':0.0,'reinspect':th['Cd_r'],'return':th['Cd_ret'],'recall':th['Cd_rec']}
    P,W,B=th['P'],th['W'],th['B']; Fm,Fs,I=th['F_m'],th['F_s'],th['I']
    Rp,Rc=th['R_p'],th['R_c']; Vq,G,Sc=th['Vq'],th['G'],th['Sc']
    payoff=np.zeros((N_PLAYERS,)+tuple([K_MAX]*N_PLAYERS))
    for idx in itertools.product(*[range(STRAT_COUNTS[i]) for i in range(N_PLAYERS)]):
        s,m,ins,dis,reg=[STRAT[PLAYERS[i]][idx[i]] for i in range(N_PLAYERS)]
        q=Q_SUP[s]*Q_MFR[m]; defect=1-q
        pdet=1-(1-D_INS[ins])*(1-D_DIS[dis])*(1-D_REG[reg])
        escaped=defect*(1-pdet); caught=defect*pdet
        qbonus=Vq*q                                   # market pays for realized quality
        sup = P + 0.5*qbonus - c_sup[s] - caught*Fs - escaped*0.5*W
        man = P + qbonus - c_mfr[m] - caught*Fm - escaped*W
        insp= Rp + Sc*_INS_LVL[ins] - c_ins[ins] + caught*0.3*Fm - escaped*B
        dist= Rc + 0.5*qbonus - c_dis[dis] - escaped*B + (I*defect if dis in ('return','recall') else 0)
        regu= G*_A_LVL[reg] - c_reg[reg] + caught*0.25*(Fm+Fs) - escaped*0.5*B
        payoff[(slice(None),)+idx]=[sup,man,insp,dist,regu]
    return jnp.array(payoff)

# quick check
_th={n:float((lo+hi)/2) for n,(lo,hi) in PARAM_BOUNDS.items()}
_pt=build_payoff_tensor(_th)
print("payoff tensor:", _pt.shape, "| entries:", _pt.size)

payoff tensor: (5, 4, 4, 4, 4, 4) | entries: 5120


In [6]:
WIDTH = 64
def init_agents(key, n_players, ctx_dim, k_max, width=WIDTH):
    keys = jax.random.split(key, n_players)
    def one(k):
        k1,k2,k3=jax.random.split(k,3)
        return dict(
            W1=jax.random.normal(k1,(ctx_dim,width))*jnp.sqrt(2/ctx_dim), b1=jnp.zeros(width),
            W2=jax.random.normal(k2,(width,k_max))*0.01, b2=jnp.zeros(k_max),
            Wv=jax.random.normal(k3,(width,1))*0.01, bv=jnp.zeros(1))
    return jax.vmap(one)(keys)

def agents_forward(params, ctx):
    def one(p):
        h=jnp.tanh(ctx @ p['W1'] + p['b1'])
        return h @ p['W2'] + p['b2'], (h @ p['Wv'] + p['bv'])[0]
    return jax.vmap(one)(params)   # (N,K_MAX), (N,)

def gumbel_argmax(key, logits, mask):
    g=-jnp.log(-jnp.log(jax.random.uniform(key,logits.shape)+1e-20)+1e-20)
    return jnp.argmax(jnp.where(mask>0, logits+g, -1e30))
print("networks defined")


networks defined


In [7]:
@partial(jax.jit, static_argnums=(6,7,8))
def run_inner(params, ctx, keys, payoff_tensor, valid_mask, target_probs,
              n_players, k_max, record_every, lr=3e-4, clip=0.2):
    def gather(actions):
        return payoff_tensor[(slice(None),)+tuple(actions[i] for i in range(n_players))]
    def step(carry, key):
        params,m,v,t=carry
        logits,vals=agents_forward(params,ctx)
        akeys=jax.random.split(key,n_players)
        actions=jax.vmap(gumbel_argmax)(akeys,logits,valid_mask)
        rewards=gather(actions)
        logp=jax.vmap(lambda l,a: jax.nn.log_softmax(l)[a])(logits,actions)
        adv=rewards-vals
        def loss(params):
            lg,vl=agents_forward(params,ctx)
            lp=jax.vmap(lambda l,a: jax.nn.log_softmax(l)[a])(lg,actions)
            r=jnp.exp(lp-logp); rc=jnp.clip(r,1-clip,1+clip)
            return -jnp.mean(jnp.minimum(r*adv,rc*adv))+0.5*jnp.mean((vl-rewards)**2)
        g=jax.grad(loss)(params)
        m=jax.tree_util.tree_map(lambda m,g:0.9*m+0.1*g,m,g)
        v=jax.tree_util.tree_map(lambda v,g:0.999*v+0.001*g*g,v,g)
        mh=jax.tree_util.tree_map(lambda m:m/(1-0.9**(t+1)),m)
        vh=jax.tree_util.tree_map(lambda v:v/(1-0.999**(t+1)),v)
        params=jax.tree_util.tree_map(lambda p,m,v:p-lr*m/(jnp.sqrt(v)+1e-8),params,mh,vh)
        probs=jax.vmap(lambda l,mm: jax.nn.softmax(jnp.where(mm>0,l,-1e30)))(logits,valid_mask)
        return (params,m,v,t+1),probs
    m0=jax.tree_util.tree_map(jnp.zeros_like,params)
    v0=jax.tree_util.tree_map(jnp.zeros_like,params)
    (params,_,_,_),hist=lax.scan(step,(params,m0,v0,0),keys)
    final=hist[-1]
    tail=hist[-record_every:]
    stab=jnp.mean(jnp.var(tail,axis=0))
    return final,stab,hist
print("inner loop compiled-ready")


inner loop compiled-ready


In [8]:
def scale_action_to_theta(a01):
    # a01 in [0,1]^N_PARAMS -> physical parameters, then repair logical ordering.
    vals = np.array(LOWER) + np.array(a01)*(np.array(UPPER)-np.array(LOWER))
    th = {n: float(vals[i]) for i,n in enumerate(PARAM_NAMES)}
    # monotone repair: sort each cost tier so the ordering constraint always holds
    for grp in _ORDER_GROUPS:                      # grp is high->low
        sorted_vals = sorted((th[n] for n in grp), reverse=True)
        for n,v in zip(grp, sorted_vals): th[n]=v
    # price must exceed the highest input cost (P > Cs_h); lift if needed
    if th['P'] <= th['Cs_h']:
        th['P'] = th['Cs_h'] + 5.0
    return th

def outer_reward(theta_dict, target_probs, key, n_episodes=2000,
                 record_every=20, w_strategy=1.0, w_stability=0.5):
    payoff = build_payoff_tensor(theta_dict)
    params = init_agents(key, N_PLAYERS, CTX_DIM, K_MAX)
    keys = jax.random.split(key, n_episodes)
    final, stab, _ = run_inner(params, CTX, keys, payoff, VALID_MASK, target_probs,
                               N_PLAYERS, K_MAX, record_every)
    # strategy loss: BCE-style match to target on valid strategies
    eps=1e-7; fp=jnp.clip(final,eps,1-eps)
    strat_loss = -jnp.sum(VALID_MASK*(target_probs*jnp.log(fp)))
    return float(-(w_strategy*strat_loss + w_stability*stab)), np.array(final)

CTX_DIM = 6
CTX = jnp.ones(CTX_DIM)/2.0     # fixed context (stateless game); hook for multi-context extension
print("outer reward defined")


outer reward defined


In [9]:
def _kaiming(key,fan_in,fan_out):
    return jax.random.normal(key,(fan_in,fan_out))*jnp.sqrt(2.0/fan_in)
def init_actor_params(key, ctx_dim, n_params, hidden=256):
    k1,k2,k3,k4=jax.random.split(key,4)
    return {'w1':_kaiming(k1,ctx_dim,hidden),'b1':jnp.zeros(hidden),
            'w2':_kaiming(k2,hidden,hidden),'b2':jnp.zeros(hidden),
            'wm':_kaiming(k3,hidden,n_params),'bm':jnp.zeros(n_params),
            'ws':_kaiming(k4,hidden,n_params),'bs':jnp.zeros(n_params)}
def init_critic_params(key, ctx_dim, n_params, hidden=256):
    inp=ctx_dim+n_params; ks=jax.random.split(key,6)
    return {'w1_q1':_kaiming(ks[0],inp,hidden),'b1_q1':jnp.zeros(hidden),
            'w2_q1':_kaiming(ks[1],hidden,hidden),'b2_q1':jnp.zeros(hidden),
            'w3_q1':_kaiming(ks[2],hidden,1),'b3_q1':jnp.zeros(1),
            'w1_q2':_kaiming(ks[3],inp,hidden),'b1_q2':jnp.zeros(hidden),
            'w2_q2':_kaiming(ks[4],hidden,hidden),'b2_q2':jnp.zeros(hidden),
            'w3_q2':_kaiming(ks[5],hidden,1),'b3_q2':jnp.zeros(1)}
def actor_forward(p,s):
    h=jax.nn.relu(s@p['w1']+p['b1']); h=jax.nn.relu(h@p['w2']+p['b2'])
    return h@p['wm']+p['bm'], jnp.clip(h@p['ws']+p['bs'],-20,2)
def actor_sample(p,s,key):
    mean,log_std=actor_forward(p,s); std=jnp.exp(log_std)
    eps=jax.random.normal(key,mean.shape); a=jnp.tanh(mean+std*eps)
    lp=(-0.5*(eps**2+jnp.log(2*jnp.pi)+2*log_std)).sum(-1,keepdims=True)
    lp-=jnp.sum(jnp.log(1-a**2+1e-6),-1,keepdims=True)
    return a,lp
def q_value(cp,ctx,a):
    sa=jnp.concatenate([ctx,a])
    h=jax.nn.relu(sa@cp['w1_q1']+cp['b1_q1']); h=jax.nn.relu(h@cp['w2_q1']+cp['b2_q1']); q1=(h@cp['w3_q1']+cp['b3_q1'])[0]
    h2=jax.nn.relu(sa@cp['w1_q2']+cp['b1_q2']); h2=jax.nn.relu(h2@cp['w2_q2']+cp['b2_q2']); q2=(h2@cp['w3_q2']+cp['b3_q2'])[0]
    return q1,q2
def init_adam(params): return {'m':jax.tree_util.tree_map(jnp.zeros_like,params),
                               'v':jax.tree_util.tree_map(jnp.zeros_like,params),'t':0}
def adam_step(params,grads,state,lr=3e-4):
    t=state['t']+1
    m=jax.tree_util.tree_map(lambda m,g:0.9*m+0.1*g,state['m'],grads)
    v=jax.tree_util.tree_map(lambda v,g:0.999*v+0.001*g*g,state['v'],grads)
    mh=jax.tree_util.tree_map(lambda m:m/(1-0.9**t),m); vh=jax.tree_util.tree_map(lambda v:v/(1-0.999**t),v)
    new=jax.tree_util.tree_map(lambda p,a,b:p-lr*a/(jnp.sqrt(b)+1e-8),params,mh,vh)
    return new,{'m':m,'v':v,'t':t}

def sac_search(target_probs, n_runs, n_steps, n_episodes, use_early_stopping=False,
               patience=1000, alpha_ent=0.2, seed0=0, verbose_every=10):
    ctx=CTX; results=[]
    for run in range(n_runs):
        key=jax.random.PRNGKey(seed0+run*777); key,ka,kc=jax.random.split(key,3)
        ap=init_actor_params(ka,CTX_DIM,N_PARAMS); cp=init_critic_params(kc,CTX_DIM,N_PARAMS)
        co=init_adam(cp); ao=init_adam(ap)
        best_r=-1e18; best_theta=None; best_profile=None; stall=0
        for step in range(n_steps):
            key,ks,ke=jax.random.split(key,3)
            a,_=actor_sample(ap,ctx,ks)
            theta=scale_action_to_theta(np.clip((np.array(a)+1)/2,0,1))
            r,prof=outer_reward(theta,target_probs,ke,n_episodes)
            def closs(cp):
                q1,q2=q_value(cp,ctx,a); return (q1-r)**2+(q2-r)**2
            cp,co=adam_step(cp,jax.grad(closs)(cp),co)
            def aloss(ap):
                aa,lp=actor_sample(ap,ctx,ks); q1,_=q_value(cp,ctx,aa)
                return alpha_ent*lp.sum()-q1
            ap,ao=adam_step(ap,jax.grad(aloss)(ap),ao)
            if r>best_r: best_r=r; best_theta=theta; best_profile=prof; stall=0
            else: stall+=1
            if use_early_stopping and stall>=patience: break
        results.append(dict(run_id=run+1,reward=best_r,optimizer='SAC',profile=best_profile,**best_theta))
        if (run+1)%verbose_every==0: print(f"  SAC run {run+1}/{n_runs}: best_r={best_r:.3f}")
    return pd.DataFrame(results)
print("SAC outer loop defined (actor/twin-critic/adam, early stopping OFF by default)")


SAC outer loop defined (actor/twin-critic/adam, early stopping OFF by default)


In [10]:
def replicator_field(state, payoff_tensor_np):
    # state: list of length N of prob vectors (len K_MAX, padded). Multi-population replicator.
    # Expected payoff to player i for each pure strategy s_i is obtained by contracting
    # payoff_tensor_np[i] over every OTHER player's mixed strategy. We contract axes from
    # highest to lowest index (skipping i) so the remaining axis indices stay valid.
    dstate=[]
    for i in range(N_PLAYERS):
        ev = payoff_tensor_np[i]                       # shape (k,)*N
        for j in reversed(range(N_PLAYERS)):
            if j==i: continue
            ev = np.tensordot(ev, state[j], axes=([j],[0]))
        ki=STRAT_COUNTS[i]
        ev=ev[:ki]; xi=state[i][:ki]
        avg=np.dot(xi,ev)
        d=xi*(ev-avg)
        full=np.zeros(K_MAX); full[:ki]=d
        dstate.append(full)
    return dstate

def simulate_replicator(theta_dict, target, n_starts=20, T=80.0, dt=0.1):
    pt=np.array(build_payoff_tensor(theta_dict))
    tgt=[np.array(target[i]) for i in range(N_PLAYERS)]
    successes=0
    for s in range(n_starts):
        rng=np.random.default_rng(1000+s)
        state=[_rand_simplex(rng,STRAT_COUNTS[i]) for i in range(N_PLAYERS)]
        for _ in range(int(T/dt)):
            d=replicator_field(state, pt)
            state=[_project(state[i]+dt*d[i], STRAT_COUNTS[i]) for i in range(N_PLAYERS)]
        # success if each player's argmax matches target argmax
        ok=all(np.argmax(state[i][:STRAT_COUNTS[i]])==np.argmax(tgt[i][:STRAT_COUNTS[i]])
               for i in range(N_PLAYERS))
        successes+=ok
    return successes/n_starts

def _rand_simplex(rng,k):
    v=rng.uniform(0,1,k); full=np.zeros(K_MAX); full[:k]=v/v.sum(); return full
def _project(v,k):
    v=np.clip(v,1e-6,None); s=v[:k].sum(); full=np.zeros(K_MAX); full[:k]=v[:k]/s; return full

print("validation (replicator + numerical Jacobian) defined")


validation (replicator + numerical Jacobian) defined


In [11]:
def numerical_jacobian_eigs(theta_dict, target, eps=1e-4):
    pt=np.array(build_payoff_tensor(theta_dict))
    # reduced coordinates: for each player use first (k_i - 1) shares
    dims=[STRAT_COUNTS[i]-1 for i in range(N_PLAYERS)]
    def to_state(zred):
        state=[]; off=0
        for i in range(N_PLAYERS):
            ki=STRAT_COUNTS[i]; parts=zred[off:off+ki-1]; off+=ki-1
            full=np.zeros(K_MAX); full[:ki-1]=parts; full[ki-1]=1-parts.sum()
            state.append(np.clip(full,1e-6,1))
        return state
    def field_red(zred):
        st=to_state(zred); d=replicator_field(st,pt); out=[]
        for i in range(N_PLAYERS):
            out.extend(d[i][:STRAT_COUNTS[i]-1])
        return np.array(out)
    # target vertex in reduced coords
    z0=[]
    for i in range(N_PLAYERS):
        ki=STRAT_COUNTS[i]; a=np.argmax(target[i][:ki])
        vec=np.zeros(ki-1)
        if a<ki-1: vec[a]=1.0
        z0.extend(vec)
    z0=np.array(z0,dtype=float)
    n=len(z0); J=np.zeros((n,n))
    f0=field_red(z0)
    for j in range(n):
        zp=z0.copy(); zp[j]+=eps; zm=z0.copy(); zm[j]-=eps
        J[:,j]=(field_red(zp)-field_red(zm))/(2*eps)
    eigs=np.linalg.eigvals(J)
    return eigs, bool(np.all(eigs.real<1e-6))
print("numerical Jacobian defined")


numerical Jacobian defined


In [12]:
# validation starts: +10 more than the configured default
N_VAL_SCAN = CFG['N_VAL_STARTS'] + 10
print("N_VAL_SCAN =", N_VAL_SCAN)

N_VAL_SCAN = 30


In [13]:
import itertools

def onehot_target(choices):
    t=np.zeros((N_PLAYERS,K_MAX))
    for i,c in enumerate(choices):
        t[i, STRAT[PLAYERS[i]].index(c)]=1.0
    return jnp.array(t)

ALL_PROFILES = list(itertools.product(*[STRAT[p] for p in PLAYERS]))
print("total pure profiles:", len(ALL_PROFILES))
print("example:", ALL_PROFILES[0])


total pure profiles: 576
example: ('high', 'fullQC', 'strict', 'accept', 'auditH')


In [ ]:
CHOICES = ['standard', 'partQC', 'sampling', 'accept', 'auditL']
LABEL   = 'B5_lean_compliant'
TARGET  = onehot_target(CHOICES)
print('target set for', LABEL, '->', CHOICES)

In [ ]:
# ==============================================================================
# SENSITIVITY ANALYSIS MODULE  (SAC branch)  — CORRECTED
#
# The SAC reward is BOTH the selection criterion AND the learning signal for the
# actor-critic. So any knob that changes the reward changes the proposals SAC
# generates, and cannot be recovered by replaying a log from a baseline run.
# Therefore:
#   - w_strategy, w_stability : REAL re-runs (reward enters actor/critic loss).
#   - inner_episodes          : REAL re-runs (reward depends on inner budget).
#   - entropy (alpha_ent)     : REAL re-runs (modifies actor loss directly).
#   - outer_steps             : FREE (prefix). best-so-far over a prefix is
#         independent of the stopping time under fixed RNG + hyperparameters;
#         nothing in the trajectory depends on when you stop. Only valid shortcut.
#
# Correctness controls:
#   * Common Random Numbers (CRN): the SAME seed set (seed0..seed0+N-1) for every
#     value of a knob, so differences are paired and reflect the knob.
#   * Every success rate carries a Wilson 95% CI and its N.
#   * STRICT pass rule: all validation replicator starts must converge.
# ==============================================================================
import numpy as np
import pandas as pd
import time, os
import jax, jax.numpy as jnp
import matplotlib.pyplot as plt


def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'), float('nan'))
    p = k / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = (z * np.sqrt(p*(1-p)/n + z*z/(4*n*n))) / denom
    return p, max(0.0, center - half), min(1.0, center + half)


def validate_theta(theta_dict, target, n_starts):
    conv = simulate_replicator(theta_dict, np.array(target), n_starts=n_starts)
    return conv, bool(conv >= 1.0)


def sac_run_best_prefix(target_probs, n_runs, n_steps, n_episodes,
                        w_strategy, w_stability, alpha_ent,
                        seed0, record_every=20, prefix_grid=None):
    """Genuine SAC search (reward drives actor/critic). Records best-so-far theta
    at every prefix length in prefix_grid (default: just n_steps). CRN via seed0."""
    if prefix_grid is None:
        prefix_grid = [n_steps]
    prefix_grid = sorted(set(int(s) for s in prefix_grid if s <= n_steps))
    ctx = CTX
    best = {S: [] for S in prefix_grid}

    for run in range(n_runs):
        key = jax.random.PRNGKey(seed0 + run*777); key, ka, kc = jax.random.split(key, 3)
        ap = init_actor_params(ka, CTX_DIM, N_PARAMS); cp = init_critic_params(kc, CTX_DIM, N_PARAMS)
        co = init_adam(cp); ao = init_adam(ap)
        running_best_r = -1e18; running_best_theta = None
        per_step_best = []

        for step in range(n_steps):
            key, ks, ke = jax.random.split(key, 3)
            a, _ = actor_sample(ap, ctx, ks)
            theta = scale_action_to_theta(np.clip((np.array(a)+1)/2, 0, 1))

            payoff = build_payoff_tensor(theta)
            params = init_agents(ke, N_PLAYERS, CTX_DIM, K_MAX)
            keys = jax.random.split(ke, n_episodes)
            final, stab, _ = run_inner(params, ctx, keys, payoff, VALID_MASK,
                                       target_probs, N_PLAYERS, K_MAX, record_every)
            eps = 1e-7; fp = jnp.clip(final, eps, 1-eps)
            strat_loss = float(-jnp.sum(VALID_MASK*(target_probs*jnp.log(fp))))
            r = -(w_strategy*strat_loss + w_stability*float(stab))

            def closs(cp):
                q1, q2 = q_value(cp, ctx, a); return (q1-r)**2 + (q2-r)**2
            cp, co = adam_step(cp, jax.grad(closs)(cp), co)
            def aloss(ap):
                aa, lp = actor_sample(ap, ctx, ks); q1, _ = q_value(cp, ctx, aa)
                return alpha_ent*lp.sum() - q1
            ap, ao = adam_step(ap, jax.grad(aloss)(ap), ao)

            if r > running_best_r:
                running_best_r = r; running_best_theta = theta
            per_step_best.append(running_best_theta)

        for S in prefix_grid:
            best[S].append(per_step_best[S-1])
    return best


def _success_from_thetas(thetas, target, n_val_starts):
    passes = 0
    for th in thetas:
        passes += validate_theta({n: float(th[n]) for n in PARAM_NAMES},
                                 target, n_val_starts)[1]
    p, lo, hi = wilson_ci(passes, len(thetas))
    return len(thetas), passes, p, lo, hi


def sweep_w_strategy(target, values, n_runs, n_steps, n_episodes,
                     n_val_starts, baseline_w_stability=0.5, alpha_ent=0.2,
                     seed0=1, verbose=True):
    rows = []
    for w in values:
        t0 = time.time()
        best = sac_run_best_prefix(target, n_runs, n_steps, n_episodes,
                                   w_strategy=w, w_stability=baseline_w_stability,
                                   alpha_ent=alpha_ent, seed0=seed0)
        n, k, p, lo, hi = _success_from_thetas(best[n_steps], target, n_val_starts)
        dt = time.time() - t0
        rows.append(dict(w_strategy=w, w_stability=baseline_w_stability,
                         n=n, n_pass=k, success_rate=p, ci_low=lo, ci_high=hi,
                         walltime_sec=dt))
        if verbose:
            print(f"  w_strategy={w:.3f}: success={p:.3f} [{lo:.3f},{hi:.3f}] n={n} ({dt:.1f}s)")
    return pd.DataFrame(rows)


def sweep_w_stability(target, values, n_runs, n_steps, n_episodes,
                      n_val_starts, baseline_w_strategy=1.0, alpha_ent=0.2,
                      seed0=1, verbose=True):
    rows = []
    for w in values:
        t0 = time.time()
        best = sac_run_best_prefix(target, n_runs, n_steps, n_episodes,
                                   w_strategy=baseline_w_strategy, w_stability=w,
                                   alpha_ent=alpha_ent, seed0=seed0)
        n, k, p, lo, hi = _success_from_thetas(best[n_steps], target, n_val_starts)
        dt = time.time() - t0
        rows.append(dict(w_strategy=baseline_w_strategy, w_stability=w,
                         n=n, n_pass=k, success_rate=p, ci_low=lo, ci_high=hi,
                         walltime_sec=dt))
        if verbose:
            print(f"  w_stability={w:.3f}: success={p:.3f} [{lo:.3f},{hi:.3f}] n={n} ({dt:.1f}s)")
    return pd.DataFrame(rows)


def sweep_inner_episodes(target, values, n_runs, n_steps,
                         n_val_starts, w_strategy=1.0, w_stability=0.5,
                         alpha_ent=0.2, seed0=1, verbose=True):
    rows = []
    for ep in values:
        t0 = time.time()
        best = sac_run_best_prefix(target, n_runs, n_steps, ep,
                                   w_strategy=w_strategy, w_stability=w_stability,
                                   alpha_ent=alpha_ent, seed0=seed0)
        n, k, p, lo, hi = _success_from_thetas(best[n_steps], target, n_val_starts)
        dt = time.time() - t0
        rows.append(dict(inner_episodes=ep, n=n, n_pass=k, success_rate=p,
                         ci_low=lo, ci_high=hi, walltime_sec=dt))
        if verbose:
            print(f"  inner_episodes={ep}: success={p:.3f} [{lo:.3f},{hi:.3f}] n={n} ({dt:.1f}s)")
    return pd.DataFrame(rows)


def sweep_outer_steps(target, values, n_runs, n_episodes,
                      n_val_starts, w_strategy=1.0, w_stability=0.5,
                      alpha_ent=0.2, seed0=1, verbose=True):
    max_steps = max(values)
    t0 = time.time()
    best = sac_run_best_prefix(target, n_runs, max_steps, n_episodes,
                               w_strategy=w_strategy, w_stability=w_stability,
                               alpha_ent=alpha_ent, seed0=seed0,
                               prefix_grid=values)
    dt = time.time() - t0
    rows = []
    for S in sorted(set(int(s) for s in values if s <= max_steps)):
        n, k, p, lo, hi = _success_from_thetas(best[S], target, n_val_starts)
        rows.append(dict(outer_steps=S, n=n, n_pass=k, success_rate=p,
                         ci_low=lo, ci_high=hi))
        if verbose:
            print(f"  outer_steps={S}: success={p:.3f} [{lo:.3f},{hi:.3f}] n={n}")
    if verbose:
        print(f"  (outer-steps swept free from one {n_runs}-run set: {dt:.1f}s)")
    return pd.DataFrame(rows)


def sweep_entropy(target, values, n_runs, n_steps, n_episodes,
                  n_val_starts, w_strategy=1.0, w_stability=0.5,
                  seed0=1, verbose=True):
    rows = []
    for ae in values:
        t0 = time.time()
        best = sac_run_best_prefix(target, n_runs, n_steps, n_episodes,
                                   w_strategy=w_strategy, w_stability=w_stability,
                                   alpha_ent=ae, seed0=seed0)
        n, k, p, lo, hi = _success_from_thetas(best[n_steps], target, n_val_starts)
        dt = time.time() - t0
        rows.append(dict(alpha_ent=ae, n=n, n_pass=k, success_rate=p,
                         ci_low=lo, ci_high=hi, walltime_sec=dt))
        if verbose:
            print(f"  entropy={ae:.3f}: success={p:.3f} [{lo:.3f},{hi:.3f}] n={n} ({dt:.1f}s)")
    return pd.DataFrame(rows)


def plot_success_curve(df, xcol, title, path, logx=False):
    if df is None or len(df) == 0 or xcol not in df.columns:
        print(f"  [skip plot] {title}: no data"); return None
    fig, ax = plt.subplots(figsize=(6, 4))
    x = df[xcol].values; y = df['success_rate'].values
    lo = df['ci_low'].values; hi = df['ci_high'].values
    ax.plot(x, y, 'o-', color='#2b6cb0', lw=2, label='success rate')
    ax.fill_between(x, lo, hi, alpha=0.2, color='#2b6cb0', label='95% Wilson CI')
    if logx:
        ax.set_xscale('log')
    ax.set_xlabel(xcol); ax.set_ylabel('convergence success rate')
    ax.set_ylim(-0.02, 1.02); ax.set_title(title); ax.grid(alpha=0.3); ax.legend()
    fig.tight_layout(); fig.savefig(path, dpi=140); plt.close(fig)
    return path


print("sensitivity module loaded (corrected: real re-runs; outer-steps free)")


In [ ]:
# ==============================================================================
# SENSITIVITY STUDY — CONFIGURATION
# ==============================================================================
SENS_PAPER_SCALE = True   # True = full study (N=30); False = quick smoke run

if SENS_PAPER_SCALE:
    SCFG = dict(
        N_RUNS       = 30,     # runs per knob value (CRN across values)
        N_OUTER      = 500,    # outer steps per SAC search
        N_EPISODES   = 3000,   # inner episodes per evaluation (baseline)
        N_VAL_STARTS = 30,     # replicator starts for STRICT validation
    )
else:
    SCFG = dict(N_RUNS=4, N_OUTER=12, N_EPISODES=600, N_VAL_STARTS=12)

# Baseline (held fixed while sweeping another knob)
BASE_W_STRATEGY = 1.0
BASE_W_STABILITY = 0.5
BASE_ALPHA_ENT = 0.2

# Knob value grids
W_STRATEGY_VALUES  = [0.25, 0.5, 1.0, 2.0, 4.0]
W_STABILITY_VALUES = [0.0, 0.25, 0.5, 1.0, 2.0]
ENTROPY_VALUES     = [0.0, 0.05, 0.1, 0.2, 0.4]

# inner-episode grid (real re-runs at each budget)
INNER_EP_VALUES = [v for v in [500, 1000, 2000, 3000] if v <= SCFG['N_EPISODES']]
if SCFG['N_EPISODES'] not in INNER_EP_VALUES:
    INNER_EP_VALUES.append(SCFG['N_EPISODES'])
INNER_EP_VALUES = sorted(set(INNER_EP_VALUES))

# outer-step grid: swept FREE from one N_OUTER-length set of runs (prefix)
_raw = [50, 100, 200, 300, 500, 1000]
OUTER_STEP_VALUES = sorted(set([s for s in _raw if s < SCFG['N_OUTER']] + [SCFG['N_OUTER']]))
if len(OUTER_STEP_VALUES) < 2:
    q = max(1, SCFG['N_OUTER'] // 4)
    OUTER_STEP_VALUES = sorted(set([q, 2*q, 3*q, SCFG['N_OUTER']]))

SENS_OUTDIR = f"sensitivity_{LABEL}"
os.makedirs(SENS_OUTDIR, exist_ok=True)

def _save(df, name):
    p = os.path.join(SENS_OUTDIR, name); df.to_csv(p, index=False)
    print(f"  saved -> {p}  ({len(df)} rows)"); return p

print("Sensitivity config:", SCFG)
print("W_STRATEGY:", W_STRATEGY_VALUES, "| W_STABILITY:", W_STABILITY_VALUES)
print("INNER_EP:", INNER_EP_VALUES, "| OUTER_STEP:", OUTER_STEP_VALUES,
      "| ENTROPY:", ENTROPY_VALUES)
print("Output dir:", SENS_OUTDIR)


## Run this task


In [ ]:
# ==============================================================================
# TASK 2/5 — stability weight (real SAC re-runs, CRN)
# ==============================================================================
import time, os
print("="*80)
print(f"SENSITIVITY [2/5] stability weight — {LABEL}  (target: {CHOICES})")
print("="*80)
_t0 = time.time()

df_wstab = sweep_w_stability(TARGET, W_STABILITY_VALUES,
    n_runs=SCFG['N_RUNS'], n_steps=SCFG['N_OUTER'], n_episodes=SCFG['N_EPISODES'],
    n_val_starts=SCFG['N_VAL_STARTS'], baseline_w_strategy=BASE_W_STRATEGY,
    alpha_ent=BASE_ALPHA_ENT, seed0=1)
_save(df_wstab, f"sensitivity_w_stability_{LABEL}.csv")

plot_success_curve(df_wstab, 'w_stability', f"{LABEL}: success vs stability weight",
                   os.path.join(SENS_OUTDIR, f"plot_w_stability_{LABEL}.png"))
print(f"\nDONE task 2 | {(time.time()-_t0)/60:.1f} min")
df_wstab
